# Which MDS embeddings of the behavioral RDM are compared with the raw one, and does embedding move behavior toward the tree?

The joint behavioral RDM (`behav_joint`, both SHINE cohorts pooled, 155 participants) is the only RDM in s2 with missing pairs: SpAM trials show about 20 images each, so some pairs are never placed together. A SMACOF embedding fills those pairs and smooths the ratings, and it is how s1 compared the cohorts' geometry. Notebooks `2_` to `6_` therefore compare the raw RDM with **two** embeddings of it, a low-dimensional one (`behav_joint_mds_low`) and a high-dimensional one (`behav_joint_mds_high`). This notebook picks their dimensionalities.

**The choice must not use the tree.** Picking the d whose embedding best matches `sem_km` and then reporting that agreement in notebooks 2 to 6 would be circular. Both dimensionalities are therefore chosen from the behavioral data alone:

* **low d: the elbow of the stress curve** (kneedle rule: the d farthest below the chord joining the two ends of the normalised curve), the most compact embedding before extra dimensions stop paying off in fit;
* **high d: held-out prediction.** Participants are split into random halves, each half's raw RDM is embedded at every d, and the embedding is scored by how well it predicts the *other* half's raw ratings. The high d is the smallest d capturing 95% of the best d's gain over the raw half.

Two embeddings rather than one, because held-out prediction keeps rising slowly up to the largest d fitted (section 3), so no single d is privileged. Agreement with the tree is shown for every d afterwards, descriptively.

RDMs: `behav_joint` (raw) and its embeddings, against `sem_km`. Run from this directory (MDS fits are cached under `analysis/results/semantic_hierarchy/mds/`; a cold run takes about 1.5 hours).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[1]  # analysis/s2_semantic_hierarchy/ -> repo root
sys.path.insert(0, str(REPO_ROOT))

from analysis.s2_semantic_hierarchy import common as c
from analysis.s2_semantic_hierarchy import mds_dimensionality as md
from analysis.utils.mds import embedding_eigenspectrum

import plotly.io as pio
pio.renderers.default = "notebook_connected"  # figures load plotly.js once instead of embedding it

In [2]:
PARTS = c.tree_parts()
RDMS = c.load_rdm_set([c.REFERENCE, c.BEHAV])
BEHAV, KM = RDMS[c.BEHAV], RDMS[c.REFERENCE]
TRIALS, N_CONDS = md.joint_trials()
REBUILT, COUNTS = md.combine(TRIALS, list(TRIALS), N_CONDS)
same = np.allclose(np.nan_to_num(REBUILT, nan=-1), np.nan_to_num(BEHAV, nan=-1))
print(f"{len(TRIALS)} participants; rebuilt RDM identical to the stored D_behav_joint_simple: {same}")

155 participants; rebuilt RDM identical to the stored D_behav_joint_simple: True


## 1. Where are the missing pairs?

If the unobserved pairs clustered at some depths of the tree, every raw behavioral estimate would weight the tree unevenly, and the embeddings, which fill exactly those pairs, would change the weighting. So coverage comes first: overall, per pooled LCA bin, and per branch-depth stratum (notebook `0_`).

In [3]:
cov = md.coverage(COUNTS, PARTS)
md.fig_coverage(cov).show()
print(cov.round(3).to_string())

                  n_pairs  frac_observed  ratings_per_observed_pair
grouping group                                                     
overall  all       262450          0.869                      2.842
LCA bin  0         127246          0.868                      2.847
         1          64696          0.869                      2.853
         2          49272          0.872                      2.823
         3          16130          0.870                      2.823
         >=4         5106          0.865                      2.812
stratum  3x3        56280          0.872                      2.826
         3x>=4     130704          0.868                      2.844
         >=4x>=4    75466          0.867                      2.849


## 2. How well does SMACOF fit at each dimensionality, and where is the elbow?

Weighted SMACOF (R `smacof`, unobserved pairs weighted 0) at d = 1 to 15, 20, 25 and 30. Stress-1 always falls with d; its elbow gives the low d.

In [4]:
FITS = md.sweep(BEHAV)
stress = md.stress_table(FITS)
D_LOW = md.choose_low(stress)
print(stress.round(4).to_string())
print(f"\nlow d (kneedle elbow over d = 1 to 30): {D_LOW}; "
      f"over d = 1 to 15 only: {md.choose_low(stress.loc[:15])}")
spectrum = embedding_eigenspectrum(FITS[30]["conf"])
print("variance share per principal axis of the d=30 embedding:", np.round(spectrum[:12], 3), "...")

      stress  niter  converged
ndim                          
1     0.5971   96.0       True
2     0.4241  250.0       True
3     0.3700  303.0       True
4     0.3483  490.0       True
5     0.3365  460.0       True
6     0.3296  477.0       True
7     0.3250  423.0       True
8     0.3221  530.0       True
9     0.3200  373.0       True
10    0.3185  378.0       True
11    0.3175  422.0       True
12    0.3166  403.0       True
13    0.3160  414.0       True
14    0.3154  424.0       True
15    0.3150  402.0       True
20    0.3138  374.0       True
25    0.3133  357.0       True
30    0.3130  341.0       True

low d (kneedle elbow over d = 1 to 30): 5; over d = 1 to 15 only: 4
variance share per principal axis of the d=30 embedding: [0.255 0.144 0.139 0.053 0.036 0.033 0.028 0.026 0.025 0.022 0.02  0.019] ...


## 3. Held-out prediction: how many dimensions does behavior support?

Five random participant splits (77 / 78 participants per half). Each half's raw RDM is embedded at d in {1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 20, 25, 30}; its fitted distances are rank-correlated with the other half's raw ratings over the pairs that half observed. Both directions are averaged within a split; the standard error is taken across the five splits. The row d = 0 is the raw half against the raw half on the same pairs: an embedding beats it only if it removes more noise than structure.

The full-sample reliability of the raw RDM (100 splits, Spearman-Brown projected) is computed here too, because notebook 2 uses it as a ceiling.

In [5]:
halves = md.split_halves(TRIALS, N_CONDS, n_splits=md.N_SPLITS_MDS)
heldout = md.heldout_prediction(halves, md.fit_halves(halves))
summary = md.heldout_summary(heldout, "spearman")
D_HIGH = md.choose_high(summary, md.HIGH_GAIN)
CHOSEN = {c.BEHAV_MDS_LOW: D_LOW, c.BEHAV_MDS_HIGH: D_HIGH}
print(summary.round(4).to_string())
print(f"\nhigh d (smallest d capturing {md.HIGH_GAIN:.0%} of the best d's held-out gain over the raw half): {D_HIGH}")
print(f"held-out Spearman at the low d ({D_LOW}): {summary.at[D_LOW, 'mean']:.4f}")
print("held-out Kendall tau-a, mean over splits:")
print(md.heldout_summary(heldout, "kendall_a")["mean"].round(4).to_string())

        mean      se
ndim                
0     0.2209  0.0009
1     0.2012  0.0063
2     0.3289  0.0028
3     0.3327  0.0012
4     0.3333  0.0012
5     0.3352  0.0009
6     0.3381  0.0008
8     0.3433  0.0009
10    0.3478  0.0006
12    0.3506  0.0007
15    0.3544  0.0007
20    0.3580  0.0007
25    0.3603  0.0007
30    0.3616  0.0007

high d (smallest d capturing 95% of the best d's held-out gain over the raw half): 20
held-out Spearman at the low d (5): 0.3352
held-out Kendall tau-a, mean over splits:
ndim
0     0.1486
1     0.1361
2     0.2230
3     0.2259
4     0.2266
5     0.2282
6     0.2304
8     0.2342
10    0.2375
12    0.2397
15    0.2424
20    0.2450
25    0.2467
30    0.2477


In [6]:
md.fig_heldout(summary, CHOSEN).show()
md.fig_stress(stress, CHOSEN).show()

In [7]:
ceiling = md.raw_ceiling(TRIALS, N_CONDS)
r_half, r_sb = ceiling["spearman"].mean(), ceiling["spearman_brown"].mean()
CEILING = md.attenuation_bound(r_sb)
print(f"raw split-half Spearman {r_half:.3f} (SD {ceiling['spearman'].std():.3f}, {len(ceiling)} splits); "
      f"Spearman-Brown full-sample reliability {r_sb:.3f}")
print(f"highest Spearman a noise-free model can reach with the raw RDM: sqrt(r_SB) = {CEILING:.3f}")

raw split-half Spearman 0.220 (SD 0.002, 100 splits); Spearman-Brown full-sample reliability 0.361
highest Spearman a noise-free model can reach with the raw RDM: sqrt(r_SB) = 0.601


## 4. Does embedding move behavior toward the tree?

Now, and only now, the tree: agreement of every embedding with `sem_km`, against the raw RDM's agreement and the ceiling from section 3. Two pair sets: the pairs the raw RDM observed (a like-for-like comparison with the raw RDM) and all pairs (including the 13% the embeddings fill in). This is descriptive: both dimensionalities were fixed above without it.

The ceiling is a bound for the *raw* RDM only. An embedding is a smoothed model of the ratings, not a noisy measurement of them, so its noise is not the raw RDM's and the bound does not apply to it.

In [8]:
km_by_d = md.km_agreement_by_dim(FITS, BEHAV, KM)
md.fig_km_by_dim(km_by_d, CHOSEN, ceiling=CEILING, stat="spearman").show()
md.fig_km_by_dim(km_by_d, CHOSEN, ceiling=None, stat="kendall_a").show()
print(km_by_d.pivot(index="ndim", columns="pairs", values=["kendall_a", "spearman"]).round(4).to_string())

      kendall_a          spearman         
pairs       all observed      all observed
ndim                                      
0           NaN   0.2314      NaN   0.3505
1        0.2056   0.2055   0.3091   0.3092
2        0.4403   0.4400   0.6481   0.6480
3        0.3795   0.3801   0.5688   0.5699
4        0.3725   0.3728   0.5581   0.5587
5        0.3718   0.3729   0.5575   0.5591
6        0.3771   0.3780   0.5649   0.5662
7        0.3817   0.3827   0.5711   0.5725
8        0.3884   0.3893   0.5807   0.5820
9        0.3921   0.3931   0.5858   0.5872
10       0.3970   0.3978   0.5926   0.5938
11       0.4011   0.4019   0.5980   0.5992
12       0.4043   0.4052   0.6024   0.6035
13       0.4077   0.4086   0.6073   0.6086
14       0.4103   0.4113   0.6109   0.6122
15       0.4126   0.4133   0.6141   0.6151
20       0.4205   0.4213   0.6254   0.6265
25       0.4249   0.4257   0.6316   0.6326
30       0.4278   0.4284   0.6357   0.6366


## 5. Do the level-stratified results depend on d?

Notebooks 3 and 4 report the two embeddings only. To show whether their conclusions would change at another d, their two summary measures are computed here at every d of section 3's grid: the depth-gradient rank slope (notebook 3; larger = distance grows more steeply as pairs diverge higher up the tree) and the nested per-level separation (notebook 4; level L contrasted only among pairs that share their level L-1 ancestor). 100 image-bootstrap draws per d, holding each MDS fit fixed (no refit per draw), so the CIs leave out fit variability.

In [9]:
from analysis.s2_semantic_hierarchy import depth_gradient as dg
from analysis.s2_semantic_hierarchy import level_separation as ls

by_d = {c.BEHAV: BEHAV, **{f"behav_joint_mds{d}": FITS[d]["confdist"] for d in md.SPLIT_DIMS}}
slopes = dg.rank_slope_table(by_d, n_boot=100, cache_dir=c.cache_path("robustness"))
nested = ls.nested_separation_table(by_d, n_boot=100, cache_dir=c.cache_path("robustness"))
md.fig_robustness(slopes, nested, CHOSEN).show()
print(slopes.round(3).to_string())
print(nested.pivot(index="rdm", columns="level", values="d").round(3).to_string())

                   rank_slope  ci_low  ci_high
rdm                                           
behav_joint             0.393   0.378    0.409
behav_joint_mds1        0.334   0.295    0.373
behav_joint_mds2        0.720   0.701    0.741
behav_joint_mds3        0.644   0.621    0.671
behav_joint_mds4        0.626   0.606    0.649
behav_joint_mds5        0.625   0.605    0.647
behav_joint_mds6        0.630   0.609    0.648
behav_joint_mds8        0.643   0.624    0.661
behav_joint_mds10       0.655   0.635    0.673
behav_joint_mds12       0.665   0.646    0.683
behav_joint_mds15       0.675   0.657    0.694
behav_joint_mds20       0.686   0.666    0.705
behav_joint_mds25       0.692   0.672    0.712
behav_joint_mds30       0.696   0.676    0.716
level                  1      2      3
rdm                                   
behav_joint        0.613  0.941  0.928
behav_joint_mds1   0.426  0.755  0.128
behav_joint_mds10  1.195  2.006  1.651
behav_joint_mds12  1.209  2.014  1.675
behav_joint_md

## Save the two embeddings

Written as `D_behav_joint_mds_low.npy` and `D_behav_joint_mds_high.npy`, each with its provenance (d, stress, selection rule) in `analysis/results/rdms/metadata.json`, so notebooks 2 to 6 load them like any other RDM.

In [10]:
md.save_chosen(FITS, c.BEHAV_MDS_LOW, D_LOW, rule="kneedle elbow of stress-1 over d = 1 to 30")
md.save_chosen(FITS, c.BEHAV_MDS_HIGH, D_HIGH,
               rule=f"smallest d capturing {md.HIGH_GAIN:.0%} of the held-out Spearman gain (5 participant splits)")
print(f"saved {c.BEHAV_MDS_LOW} (d = {D_LOW}) and {c.BEHAV_MDS_HIGH} (d = {D_HIGH})")

saved behav_joint_mds_low (d = 5) and behav_joint_mds_high (d = 20)


## 6. Takeaways

1. **The missing pairs are spread evenly over the tree.** 86.9% of pairs are observed overall, and every pooled LCA bin and branch-depth stratum lies between 86.5% and 87.2%, at about 2.8 ratings per observed pair. The raw behavioral RDM does not weight any level of the tree more than another.
2. **Every SMACOF fit converged**; stress-1 falls from 0.597 (d = 1) to 0.337 (d = 5), 0.314 (d = 20) and 0.313 (d = 30). The kneedle elbow is d = 5 over d = 1 to 30 (d = 4 if only d = 1 to 15 is considered), so the low-d choice is somewhat sensitive to the range searched.
3. **Behavior supports many dimensions, with diminishing returns.** An embedded half predicts the other half's raw ratings better than the raw half itself (held-out Spearman 0.221) from d = 2 on (0.329), rising slowly to 0.362 at d = 30 with no interior optimum. Hence two embeddings rather than one: **low d = 5** (held-out 0.335) and **high d = 20** (0.358, the smallest d capturing 95% of the gain). Both were fixed without reference to the tree.
4. **Embedding moves behavior strongly toward the tree.** Spearman with `sem_km` rises from 0.351 (raw) to 0.558 at d = 5 and 0.625 at d = 20 (tau-a 0.231, 0.373, 0.421), and peaks at d = 2 (0.648). Values on the raw RDM's observed pairs and on all pairs differ by at most 0.002, so the gain does not come from filling in the missing 13%. It comes from smoothing: the embedding discards rating noise that does not follow the tree.
5. **The raw reliability ceiling (0.601) does not bound the embeddings,** and the high-d embedding exceeds it. The ceiling describes the raw RDM's measurement noise; an embedding is a smoothed model of the ratings. Its higher agreement is a property of the model, not evidence that behavior is more reliable than its split-half reliability says. Notebooks 2 to 6 therefore report raw and embedded behavior side by side.
6. **The level-stratified measures barely depend on d once d >= 3.** The depth-gradient rank slope is 0.63 at d = 5 and 0.69 at d = 20 (raw 0.39), and lies between 0.63 and 0.70 for every d from 3 to 30. The nested separation peaks at the mid level at every d (d = 5: 1.16, 2.06, 1.53 for levels 1 to 3; d = 20: 1.25, 2.07, 1.66; raw 0.61, 0.94, 0.93). Only d = 1 and 2 behave differently (d = 2 maximises the coarse levels). Conclusions drawn at d = 5 and 20 should hold anywhere in d = 3 to 30.

Limitations: the half-sample fits use five participant splits, which share participants, so the standard errors understate between-sample variability. The image-bootstrap CIs of section 5 hold each MDS fit fixed. The embedding imposes Euclidean structure, which favours compact clusters; part of its agreement with the tree may reflect that structure rather than the ratings alone.